# CIBUSmod example
This notebook privides a step-by-step example of how the model is run and how outputs can be visualised. The model is currently a work in progress, so this notebook will be continuosly updated as the work progresses.

## Setting things up

### Import libraries
Add directory with the CIBUSmod modules to path to be able to import

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(),'..'))

Import CIBUSmod and packages for handling data and plotting

In [2]:
import CIBUSmod as cm
import CIBUSmod.utils.plot as plot
from CIBUSmod.soil.icbm_funcs import set_df_and_name

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root: /home/niceri/Pythoncode/CIBUSmod/notebooks/../CIBUSmod/..
input_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/input
temp_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
export_path: /home/niceri/Pythoncode/CIBUSmod/data/soil/exported_results


## Run scenario 'Food as industry'

### Set up scenario and modules

Instantiate a `Session` with a `name` and `data_path` to the folder with input data. Next add scenarios to run with `.add_scenario()`. Here we use a scenario defined in `data/scenarios/food_as_industry.xlsx` and calculate outputs for 5-year time-steps from 2020 to 2050.

In [3]:
# Create session
session = cm.Session(
    name = 'food_as_industry',
    data_path = '../data'
)

# Define scenarios
session.add_scenario(
    name = 'FAI',                           # Name of scenario in outputs
    scenario = 'food_as_industry',          # Name of scenario Excel file
    modules = 'all',                        # Modules to update
    pars = 'all',                           # Parameters to update
    years = ['2020', '2035', '2050']        # Years to calculate
)
# session.add_scenario(
#     name = 'FAI_no_cows',
#     # Here a list of scenarios are provided. These are handled in consequtive order
#     # and if the same parameter is updated in sevral scenarios only the latest in
#     # the list will have an effect.
#     scenario = ['food_as_industry', 'no_cows'], 
#     modules = 'all',
#     pars = 'all',
    # years = ['2020', '2035', '2050']
# )

A scenario with the name 'FAI' already exists use .update_scenario() or .remove_scenario() instead.


Next, we initialise all model modules (python classes) that handle the calculations.

Each module is intialised with a `ParameterRetriever` object which reads a named parameter Excel-file from the default data folder and handles the retrieval of parameter values used in the calculations. The `ParameterRetriever` also handles updating parameter values according to a scenario.

There are four main modules that store data as attributes (generaly in the form of pandas.DataFrames). These are:

`Regions`
This module handles baselina crop areas and animal numbers ('x0') as well as various regional attrubutes such as climate and soil properties.

`DemandAndConversions`
This module calculates demand for crop and animal production based on population, consumption of different foods, waste and conversion factors, etc. It also calculates waste and by-products generated.

`CropProduction`
This module calculates production of crop products per unit (area) of a certain crop in a certain region.

`AnimalHerd`
This module has subclasses for each animal species (and breed) and is initialised as one AnimalHerd object per combination of species (e.g. cattle), breed (e.g. dairy), production system (e.g. conventional) and sub system (used to represent different feeding strategies, e.g. maize based). The `AnimalHerd` modules calculates production of animal products per animal unit (i.e. a defining animal in the species/breed). The defining animal differs accros species/breeds with e.g. 'cows' for `CattleHerd`s, 'sows+gilts' for `PigHerd`s and 'total horses' for `HorseHerd`s.

There are also a number of management (mgmt) modules that calculates specific aspects on the main modules. These have no data attributes but do have their own parameter Excel-files connected to them. These are:

`FeedMgmt`
Handles the calculation of feed requirements, losses and import shares and the translation from 'feed products' to 'crop products' and 'by-products'.

`ManureMgmt`
Handles the calculation of manure excretion and losses in stables and storage.

`PlantNutrientMgmt`
Handles the calculation of crop fertiliser requirements (N, P and K) and application of manure and mineral fertiliser.

`MachineryAndEnergyMgmt`
Handles the calculation of energy requirements in machinery, drying, greenhouses, stables etc.

`InputsMgmt`
Handles the calculation of supply chain emissions for inputs (currently energy and fertilisers) by retrieving lifecycle inventory data from ecoinvent.

Finaly the module `GeoDistributor` handles the distribution of crop and animal production across regions by solving a convex optimisation problem that minimises the deviation of crop areas and animal numbers for the current situation while meeting demand for crop and animal products as wel as a number of additional constraints.

In [4]:
# Instatiate Regions
regions = cm.Regions(
    par = cm.ParameterRetriever('Regions')
)

# Instantiate DemandAndConversions
demand = cm.DemandAndConversions(
    par = cm.ParameterRetriever('DemandAndConversions')
)

# Instantiate CropProduction
crops = cm.CropProduction(
    par = cm.ParameterRetriever('CropProduction'),
    index = regions.data_attr.get('x0_crops').index
)    

# Instantiate AnimalHerds
# Each AnimalHerd object is stored in an indexed pandas.Series
herds = cm.make_herds(regions)

# Instantiate feed management
feed_mgmt = cm.FeedMgmt(
    herds = herds,
    par = cm.ParameterRetriever('FeedMgmt')
)

# Instantiate manure management
manure_mgmt = cm.ManureMgmt(
    herds = herds,
    feed_mgmt = feed_mgmt,
    par = cm.ParameterRetriever('ManureMgmt'),
    settings = {
        'NPK_excretion_from_balance' : True
    }
)

# Instantiate crop residue managment
crop_residue_mgmt = cm.CropResidueMgmt(
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('CropResidueMgmt')
)

# Instantiate plant nutrient management
plant_nutrient_mgmt = cm.PlantNutrientMgmt(
    demand = demand,
    regions = regions,
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('PlantNutrientMgmt')
)

# Instatiate machinery and energy management
machinery_and_energy_mgmt  = cm.MachineryAndEnergyMgmt(
    regions = regions,
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('MachineryAndEnergyMgmt')
)

# Instatiate inputs management
inputs = cm.InputsMgmt(
    demand = demand,
    crops = crops,
    herds = herds,
    par = cm.ParameterRetriever('InputsMgmt')
)

# Instantiate geo distributor
geodist = cm.GeoDistributor(
    regions = regions,
    demand = demand,
    crops = crops,
    herds = herds,
    feed_mgmt = feed_mgmt,
    par = cm.ParameterRetriever('GeoDistributor')
)

### Run calculations
Now we have initialised all modules and can do the calculations. This is done by looping through scenarios and years via `Session.iterate()` and first updating all parameter values according to the speciefied scenario and year via `ParameterRetriever.update_all_parameter_values()` and then (re-)calculating all modules. After all main modules have been (re-)calculated we run `GeoDistributor.make()` and `GeoDistributor.solve()` to make and solve the optimisation problem that distribute crops and animals across regions. Then some mgmt modules are calculated and finally ouputs are stored via `Session.store()` before the next iteration.

In [ ]:
# Set to true to print progress messages
msg = False

# Loop through scenarios and years
for scn, year in session.iterate('all'):#
    print(scn,year)
    
    # Update all parameter values
    cm.ParameterRetriever.update_all_parameter_values(
        **session[scn],
        year = year
    )
    
    # Get region attributes
    regions.calculate(verbose=msg)
    
    # Calculate food demand
    demand.calculate(verbose=msg)
    
    # Calculate crops
    crops.calculate(
        verbose=msg
    )
    
    # Calculate herds
    for h in herds:
        h.calculate(verbose=msg)
    
    # Calculate feed
    feed_mgmt.calculate(verbose=msg)    
    
    # Distribute animals and crops
    # Make optimisation problem
    geodist.make(use_cons=[1,2,3,4,5,6,7], scale_power=0.4, verbose=msg)
    # Solve optimisation problem (move to next scn/year if it fails)
    try:
        geodist.solve(
            verbose=msg,
            solver_settings = {
                'solver':'OSQP',
                'max_iter':200000,
                'eps_abs':5e-6,
                'eps_rel':5e-6,
                'verbose':False
            }
        )
    except Exception as e:
        print(f'(!!!) GeoDistributor failed for {scn}, {year} with the exception: {e}')
        continue
    
    # Redistribute feeds (not yet implemented) and calculate enteric CH4 emissions
    feed_mgmt.calculate2(verbose=msg)
    
    # Calculate manure
    manure_mgmt.calculate(verbose=msg)
    
    # Calculate harvest of crop residues
    crop_residue_mgmt.calculate(verbose=msg)
    
    # Calculate plant nutrient management
    plant_nutrient_mgmt.calculate(verbose=msg)
    
    # Calculate energy requirements
    machinery_and_energy_mgmt.calculate(verbose=msg)
    
    # Calculate inputs supply chain emissions
    inputs.calculate(verbose=msg)
    
    # Store results
    session.store(
        scn, year,
        demand, regions, crops, herds
    )

### Look at the data

Use `Session` to get a summary of scenarios and modules and their data attributes.

In [ ]:
session

Use `Session.get_attr()` to retrieve specific data for all scenarios and years.

In [ ]:
session.get_attr(
    module='c',
    attr='area',
    groupby={'crop':'land_use'}
)/1000

In [ ]:
session.get_attr(
    module='A',
    attr='heads',
    groupby='species'
)/1000

## Plot output
Here are some example output plots.

In [ ]:
style = {
    'kind' : 'bar',
    'cmap' : 'YlOrBr',
    'edgecolor' : 'grey',
    'legend' : False,
    'width' : 0.7
}

for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))
    
    plot_data = (
        session.get_attr(
            module='D',
            attr='food_demand_to_processing',
            groupby=['origin','food_group']
        ).loc[scn]
        .stack()
    )/1000000
    plot_data.loc[:,'imported'] = -plot_data.loc[:,'imported']
    
    fig, ax = plt.subplots(figsize=(7,5))
    plot_data['domestic'].unstack('year').plot(**style, ax=ax)
    (
        plot_data['imported'].unstack('year')
        .rename({x:'_' for x in plot_data.index.get_level_values('year').unique()}, axis=1)
    ).plot(**style, ax=ax, alpha=0.5)
    
    plt.hlines(0,-1,11, color='black', linewidth=1)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.set_ylabel('1000 tonnes per year', size=14)
    ax.set_title('Domestic (pos.) and imported (neg.) food demand', size=12)
    ax.set_xlabel('')
    ax.legend(loc='center left', ncol=1, bbox_to_anchor=(1, 0.5), fontsize=12)
    
    # plt.tight_layout()
    plt.show()

In [ ]:
style = {
    'kind' : 'bar',
    'stacked' : True,
    'cmap' : 'Pastel1',
    'edgecolor' : 'grey',
    'legend' : False,
    'width' : 0.5
}

# Get data on GHG emissions
GHG_data = cm.get_GHG(session)

for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))

    fig, axs = plt.subplots(3,2, figsize=(10,15))

    # Land use --->
    plot_data = session.get_attr('C','area',{'crop':'crop_group2'}).loc[scn]/1000000
    
    # Cropland
    ax = axs[0,0]
    plot_data.drop(['Semi-natural grasslands','Greenhouse crops'], axis=1).plot(**style, ax=ax)
    ax.set_ylabel('Cropland area [Mha]')
    
    # Semi-natural grassland
    ax = axs[0,1]
    plot_data.xs('Semi-natural grasslands', axis=1).plot(**style, ax=ax)
    ax.set_ylabel('Semi-natural grassland area [Mha]')
    
    # Greenhouse
    ax = axs[1,0]
    plot_data.xs('Greenhouse crops', axis=1).plot(**style, ax=ax)
    ax.set_ylabel(r'Greenhouse area [million $m^2$]')

    # Mineral N use --->
    ax = axs[1,1]
    (session.get_attr('C','fertiliser.mineral_N',{'crop':'crop_group2'})/1000000).loc[scn].plot(**style, ax=ax)
    ax.set_ylabel('Mineral N use [1000 tonnes N]')

    # Energy use --->
    ax = axs[2,0]
    pd.concat([
        session.get_attr('C','energy_use','activity').loc[scn]/1000000,
        session.get_attr('A','energy_use','activity').loc[scn]/1000000
    ], axis=1).plot(**style, ax=ax)
    ax.set_ylabel('Energy use [GWh]')
    
    # GHG emissions
    ax = axs[2,1]
    plot_data = (GHG_data/1000000).loc[scn]
    (
        plot_data
        .T.groupby('process').sum().T
    ).plot(**style, ax=ax)
    ax.set_ylabel(r'GHG emissions [1000 tonnes $CO_{2}-eq$]')
    
    for ax in axs.flatten():
        ax.legend(loc='upper center', ncol=2, bbox_to_anchor=(0.5, -0.2), fontsize=10)
        ax.set_xlabel('')

    fig.tight_layout()
    plt.show()

In [ ]:
style = {
    'kind' : 'bar',
    'stacked' : True,
    'cmap' : 'Pastel1',
    'edgecolor' : 'grey',
    'legend' : False,
    'width' : 0.5
}

for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))
    
    # Number of animal heads
    plot_data = session.get_attr('A','heads',['species','breed']).loc[scn]

    fig, axs = plt.subplots(1,4, figsize=(14,5))

    for ax,sp in zip(axs,plot_data.columns.get_level_values('species').unique()):

        (
            (plot_data / (1000000 if sp=='poultry' else 1000))
            .xs(sp, level='species', axis=1)
            .plot(**style, ax = ax)
        )

        ax.set_ylabel(sp.capitalize() + (' [Million heads]' if sp=='poultry' else ' [1000 heads]'))
        ax.set_xlabel('')
        ax.legend(loc='lower left', bbox_to_anchor=(0, 0))

    for ax in axs.flatten():
        ax.legend(loc='upper center', ncol=1, bbox_to_anchor=(0.5, -0.2), fontsize=11)
        ax.set_xlabel('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))
    
    y0 = session.scenarios()[scn][0]
    yend = session.scenarios()[scn][-1]

    # ABSOLUTE
    fig, axs = plt.subplots(1,4, figsize=(10,4))
    
    # LAND USE -------->
    plot_data = session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn]/1000
    plot_data = plot_data.loc[yend] - plot_data.loc[y0]
    # Cropland area --->
    ax = axs[0]
    plot.map_from_series(
        plot_data.loc['cropland'],
        cmap='RdBu',
        vmin=-5,
        vmax=5,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Cropland area\n[1000 ha]')
    ax.set_axis_off()

    # Grassland area --->
    ax = axs[1]
    plot.map_from_series(
        plot_data.loc['semi-natural grasslands'],
        cmap='RdBu',
        vmin=-1,
        vmax=1,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Semi-natural grassland area\n[1000 ha]')
    ax.set_axis_off()

    # Manure N application --->
    plot_data = session.get_attr('C','fertiliser.manure_N',{'crop':'land_use','region':None}).loc[scn,'cropland'] \
    / session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn,'cropland']
    plot_data = plot_data.loc[yend] - plot_data.loc[y0]
    
    ax = axs[2]
    plot.map_from_series(
        plot_data,
        cmap='RdBu',
        vmin=-25,
        vmax=25,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Manure N \n[kg N/ha]')
    ax.set_axis_off()

    # Mineral N application --->
    plot_data = session.get_attr('C','fertiliser.mineral_N',{'crop':'land_use','region':None}).loc[scn,'cropland'] \
    / session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn,'cropland']
    plot_data = plot_data.loc[yend] - plot_data.loc[y0]
    
    ax = axs[3]
    plot.map_from_series(
        plot_data,
        cmap='RdBu',
        vmin=-25,
        vmax=25,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Mineral N \n[kg N/ha]')
    ax.set_axis_off()

    
    plt.suptitle('Absolute change')
    plt.tight_layout()
    plt.show()

    # PERCENT
    fig, axs = plt.subplots(1,4, figsize=(10,4))
    
    # LAND USE -------->
    plot_data = session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn]/1000
    plot_data = (plot_data.loc[yend] - plot_data.loc[y0]) / plot_data.loc[y0] * 100
    # Cropland area --->
    ax = axs[0]
    plot.map_from_series(
        plot_data.loc['cropland'],
        cmap='RdBu',
        vmin=-100,
        vmax=100,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Cropland area\n[%]')
    ax.set_axis_off()

    # Grassland area --->
    ax = axs[1]
    plot.map_from_series(
        plot_data.loc['semi-natural grasslands'],
        cmap='RdBu',
        vmin=-100,
        vmax=100,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Semi-natural grassland area\n[%]')
    ax.set_axis_off()

    # Manure N application --->
    plot_data = session.get_attr('C','fertiliser.manure_N',{'crop':'land_use','region':None}).loc[scn,'cropland'] \
    / session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn,'cropland']
    plot_data = (plot_data.loc[yend] - plot_data.loc[y0]) / plot_data.loc[y0] * 100
    
    ax = axs[2]
    plot.map_from_series(
        plot_data,
        cmap='RdBu',
        vmin=-100,
        vmax=100,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Manure N \n[%]')
    ax.set_axis_off()

    # Mineral N application --->
    plot_data = session.get_attr('C','fertiliser.mineral_N',{'crop':'land_use','region':None}).loc[scn,'cropland'] \
    / session.get_attr('C','area',{'crop':'land_use','region':None}).loc[scn,'cropland']
    plot_data = (plot_data.loc[yend] - plot_data.loc[y0]) / plot_data.loc[y0] * 100
    
    ax = axs[3]
    plot.map_from_series(
        plot_data,
        cmap='RdBu',
        vmin=-100,
        vmax=100,
        edgecolor='grey',
        ax=ax
    )
    ax.set_title('$\Delta$ Mineral N \n[%]')
    ax.set_axis_off()
   
    
    plt.suptitle('Percent change')
    plt.tight_layout()
    plt.show()

In [ ]:
selection = {
    'Dairy cattle':('cattle','dairy'),
    'Beef cattle':('cattle','beef'),
    'Horses':('horses',slice(None)),
    'Sheep':('sheep',slice(None)),
    'Pigs':('pigs',slice(None)),
    'Broiler poultry':('poultry','broiler'),
    'Layer poultry':('poultry','layer')
}

for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))
    
    plot_data = session.get_attr('A','heads',['species','breed','region']).loc[scn]/1000
    plot_data_abs = (plot_data.loc['2050'] - plot_data.loc['2020']).sort_index()
    plot_data_prc = ((plot_data.loc['2050'] - plot_data.loc['2020']) / plot_data.loc['2020'] * 100).sort_index().fillna(0)
    
    # ABSOLUTE
    fig, axs = plt.subplots(1,len(selection), figsize=(len(selection)*2.5,4))
    for ax,sel in zip(axs,selection):
        d = plot_data_abs.loc[selection[sel]]
        lim = max(abs(d))
        plot.map_from_series(
            d,
            ax=ax,
            cmap='RdBu',
            vmin=-lim*1.1,
            vmax=lim*1.1,
            edgecolor='grey'
        )
        ax.axis('off')
        ax.set_title(sel)  
    plt.suptitle('Absolute change 2020-2050 [1000 heads]')
    plt.tight_layout()
    plt.show()
    
    # PERCENT
    fig, axs = plt.subplots(1,len(selection), figsize=(len(selection)*2.5,4))
    for ax,sel in zip(axs,selection):
        d = plot_data_prc.loc[selection[sel]]
        lim = max(abs(d))
        plot.map_from_series(
            d,
            ax=ax,
            cmap='RdBu',
            vmin=-100,
            vmax=100,
            edgecolor='grey'
        )
        ax.axis('off')
        ax.set_title(sel)
    plt.suptitle('Relative change 2020-2050 [%]')
    plt.tight_layout()
    plt.show()

# Soil carbon and climate impact calculations
The remaining code calculates and manages C flows to and from soils as well as climate impacts.

In this section the `xarray` package is being heavily used for its excellent handling of multidimensional data.
It may take some time to get used to how to use it. For that reason, below is a cell with tips on how to learn using its functionalities.

For the functions and methods in the `SoilData` class to work the `xarray` package must be installed (pip install xarray)

#### The `xarray` package
`xarray` build on the `pandas` package and extends its capabilities.
There are however some differences in how data is selected that can be confusing if you are already familiar with `pandas`.
Whether you are familiar with `pandas` or not, you are in luck. `xarray`is very well documented. 
- **Tutorial:** there is a [very good tutorial](https://tutorial.xarray.dev/intro.html) that gves you an introductio. to what the package can do and how to do it.
- **Official documentation:** the [official documentation](https://docs.xarray.dev/en/stable/index.html) is very extensive and complete, with examples on how to use commands. Anything you don't find in the Tutorial you will find here.
- **Youtube:** If audio-visual learning is your cup of tea, [YouToBe](https://www.youtube.com/results?search_query=How+to+create+a+NetCDF+file+using+Python+xarray+for+beginners+-+a+depth+profile) can offer you plenty of help
- **ChatGPT:** the free version of chatGPT can help you debugging and solve coding issues, but if you have a paid account you can also access [Codex](https://openai.com/blog/openai-codex) which is excellent at assisting you in coding and will quickly get you up to speed on how to use `xarray`, or solve any other python related issue.

## Using the ouput from the session module to calculate soil carbon stock changes
First we create a dataframe with all the input fractions contributing with C to soils.  

In [4]:
from CIBUSmod.utils.output_data_manip_db import to_ICBM
icbm = to_ICBM(session)

## Create an instance of the SoilData class for each scenario
A new instance should be used for each scenario to avoid extremely long output dataseries which can cause jupyter to crash from running out of memory.

In this example 'FAI_soil' is created using the icbm dataframe. The scenario name is taken from the first row of the index level 'scn', assuming it has been set and that the dataframe does not contain multiple scenarios. 

In [5]:
soil_input_df, scn_name = set_df_and_name(icbm, session)
FAI_soil = cm.SoilData('FAI_soil', soil_input_df, scn_name)

The following code block can be used to check which variables have currently been set.
Both public and private attributes are shown to facilitate quick troubleshooting. 

In [9]:
FAI_soil.check_attributes_status('public'), FAI_soil.check_attributes_status('private') 

(('The following attributes are set for public variables',
  {'co2_fluxes': ('NoneType', False),
   'historic_ha_df': ('DataFrame', True),
   'historic_inventory': ('Dataset', True),
   'historic_sko_df': ('DataFrame', True),
   'input_inventory': ('Dataset', True),
   'name': ('str', True),
   'scenario': ('str', True),
   'soc_ha_df': ('DataFrame', True),
   'soc_inventory': ('Dataset', True),
   'soc_sko_df': ('DataFrame', True),
   'ss_input_df': ('DataFrame', True),
   'startyear': ('int32', True)}),
 ('The following attributes are set for private variables',
  {'_c_input_ha_df': ('DataFrame', True),
   '_c_input_sko_df': ('DataFrame', True),
   '_h_value_dict': ('dict', True),
   '_input_grouped_df': ('DataFrame', True),
   '_residue_col_name': ('str', True),
   '_scn_ha_sel': ('list', True),
   '_scn_sko_sel': ('list', True),
   '_spinup_groupby_df': ('DataFrame', True),
   '_ss_ha_sel': ('list', True),
   '_ss_input_ha_df': ('DataFrame', True),
   '_ss_input_sko_df': ('DataFram

## If scenario datasets have been calculated previously these can be loaded using the `load_inventory` and `load_instance_state` methods

To do so, proceed with the following cell and choose set the dataset  you wish to load (default: all `['input', 'soc', 'historic']`) in the `load_inventory` method. This will load all dataframes and datasets calculated in an earlier session

`load_instance_state` loads and sets non-dataset and non-dataframe variables from a previous session. It is a class instance and returns an instance. It must be called with the name of the pickle file previously saved to correctly identify and load the file.

In [ ]:
# The commands in this cell saves the current state of the SoilData instance
FAI_soil.save_inventory()
FAI_soil.save_instance_state()

In [3]:
FAI_soil = cm.SoilData.load_instance_state('FAI_soil')
FAI_soil.load_inventory()

dataset set to ['inputs', 'soc', 'historic']
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
Scenario dataset loaded from FAI_input_ds.nc in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
SOC dataset loaded from FAI_soc_ds.nc in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
Historic SOC dataset loaded from historic_soc_ds.nc 
 in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results


### Calculate the carbon inputs for each input in the input_df
The `calc_scn_inputs` method calculates the C input for each fraction of `input_df` and sets the  following variable attributes:
- `startyear`: defined as the first year of the timeseries in `input_df`.
- `input_inventory`: xarray dataset with `scn, crop, prod_system, region, input_year` as coordinates, including all original input and the C input per ha as well as per SKO for all fractions.
- `ss_input_df.columns`: dataframe containing the SOC inputs to use for the spinup modelling, corresponding to the C inputs for all fraction in the `startyear`, also both per ha and per SKO. 

**Note:** the column or index level name `year` in the `input_df` will be renamed to `input_year` to distinguish it from other time coordinates calculated with ICBM and subsequent temperature response functions.

In [8]:
FAI_soil.calc_scn_inputs(verbose=False)

Calculating scenario inputs...
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)


/home/niceri/Pythoncode/CIBUSmod/notebooks/../CIBUSmod/soil/soil_utils.py:514: RuntimeWarning: invalid value encountered in scalar multiply
  i_ag_1 = (param_df.iloc[:,0][crop] + param_df.iloc[:,1][crop] * H)
/home/niceri/Pythoncode/CIBUSmod/notebooks/../CIBUSmod/soil/soil_utils.py:515: RuntimeWarning: invalid value encountered in scalar multiply
  i_ag_2 = + (param_df.iloc[:,2][crop] + param_df.iloc[:,3][crop] * H)
/home/niceri/Pythoncode/CIBUSmod/notebooks/../CIBUSmod/soil/soil_utils.py:517: RuntimeWarning: invalid value encountered in scalar multiply
  i_bg = + (param_df.iloc[:,4][crop] + param_df.iloc[:,5][crop] * H)


### Calculate the SOC timeseries
The `calc_soc_timeseries` currently calculates the soc timeseries for each individual yearly input in the `input_inventory`. This is a memory intensive operation. It may cause the jupyter instance to crash if memory allocation is not sufficiently large.

**TODO:** Redefine function to operate on fractions of the database to reduce memory load.

In [9]:
FAI_soil.calc_soc_timeseries(verbose=True)

Calculating SOC timeseries...
---Executing _calculate_soc()---
---Executing _make_scn_area_dfs()---
---Leaving _make_scn_area_dfs()---
'_c_input_ha_df' and '_c_input_sko_df' generated
-> 'h_value_dict' not set.
---Executing h_map_helper()---
An h-value mapping dataframe does not exist.
Creating h_map_df from 'h_values.csv' in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
CIBUS crop mapping dataframe does not exist
> Calling 'crop_map_helper()'
---Executing crop_map_helper()---
No input dataframe exists.
Creating 'input_df' from 'crop_carbon_map.csv'in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
---Leaving crop_map_helper()---
'crop_in_df' and 'crop_re_dict' created
An amendment mapping dataframe does not exist
Creating 'amnd_map_df' from 'amnd_map.csv' in /home/niceri/Pythoncode/CIBUSmod/data/soil/input
'h_value_df' and 'h_value_dict' created
---Leaving h_map_helper()---
Extracting filtered_namelist per ha
Extracting filtered_namelist per sko
---Leaving _calculate_soc()---
---E

### Calculate the historic SOC timeseries
The `calc_historic_soc_timeseries` calculates the soc timeseries for each individual SS input in the `ss_input_df`. The inputs all take place in the year 2020 (and only this year), making the computation much lighter and faster then the previous ones.


In [10]:
FAI_soil.calc_historic_soc_timeseries(verbose=True)

Calculating historic SOC timeseries...
---Executing _calculate_historic_soc()---
> One or both of '_ss_input_ha_df' and '_ss_input_sko_df' are unset
info: Generating 'spinup_ha_df' and 'spinup_sko_df' from 'input_df'
---Executing _make_spinup_area_dfs()---
---Leaving _make_spinup_area_dfs()---
'_ss_input_ha_df' and '_ss_input_sko_df' set using input_df for FAI
Extracting filtered_namelist per ha
Extracting filtered_namelist per sko
---Leaving _calculate_historic_soc()---
---Executing _historic_icbm_calculations()---
info: Calculating SOC SS values per ha in 2020
info: Calculating SOC SS values per sko in 2020
info: Calculating SOC timeseries per ha
info: Calculating SOC timeseries per sko
info: Finished calculating historic SOC timeseries
Creating xarray historic soc dataset
---_historic_icbm_calculations() executed succesfully---


## Calculate and assign the tot SOC and CO2 fluxes
When both future and historic SOC timeseries have been calculated for a scenario the following methods have to be run to calculate the total SOC and CO2 fluxes for each year.
Both will be addaded as new dataarrays to the existing `soc_inventory` dataset

In [10]:
FAI_soil.add_total_soc()
FAI_soil.add_co2_flux()

## Saving and loading datasets
The `SoilData` class has instance methods to save and read saved datasets to avoid having to rerun the computations between sessions. These are **NOT** automatic, but have to be invoked by the user.

The `save_inventory` and `load_inventory` methods can be called with three optional strings:

- `inputs` saves and loads the input inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The  netcdf file is named `<scenario_name>_input_ds.nc`
- `soc` saves and loads the soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_soc_ds.nc`
- `historic` saves and loads the historic soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_historic_soc_ds.nc`

The `save_instance_state` saves all non dataframe/dataset variables in a pickle file. This needs to be run to include all other set variable states. Without them many methods will not work.  



In [11]:
# The commands in this cell saves the current state of the SoilData instance
FAI_soil.save_inventory()
FAI_soil.save_instance_state()

dataset set to ['inputs', 'soc', 'historic']
Scenario dataset saved as FAI_input_ds.nc in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
SOC dataset saved as FAI_soc_ds.nc in /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results


AttributeError: 'SoilData' object has no attribute 'historic_ha_df'

In [3]:
# The commands in this cell instantiates and sets all the variable states of the instance to what it was when it was previously saved
FAI_soil = cm.SoilData.load_instance_state('FAI_soil')
FAI_soil.load_inventory()

df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
The following dataset variables have been set:
 input_inventory
The following dataframe variables have been set:
 ss_input_df
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
The following dataset variables have been set:
 soc_inventory
The following dataframe variables have been set:
 soc_ha_df
 soc_sko_df
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
df_dir /home/niceri/Pythoncode/CIBUSmod/data/soil/temp_results
The following dataset variables have been set:
 historic_inventory
The following dataframe variables have been set:
 historic_ha_df
 historic_sko_df


# The cells below can be used to plot, summarise and save icbm (dataframe) data.
They are not used for any calculations

In [ ]:
for scn in session.scenarios():
    print(scn)
    print('-'*len(scn))

    fig, axs = plt.subplots(1,4, figsize=(16,5), dpi=80)

    ax=axs[0]
    (icbm.loc[scn].groupby(['year']).sum()['harvest_kgdm']/1000000).plot(kind='area', color='grey', ax=ax)
    start, end = ax.get_xlim()
    ax.set_title('harvest (1000 tonnes DM)')
    
    ax=axs[1]
    (icbm.loc[scn].groupby(['year']).sum()['crop_residues_harvest_kgdm']/1000000).plot(kind='area', color='grey', ax=ax)
    start, end = ax.get_xlim()
    ax.set_title('crop residues harvest (1000 tonnes DM)')
    
    ax=axs[2]
    (icbm.loc[scn].groupby(['year']).sum()['area_ha']/1000).plot(kind='area', color='grey', ax=ax)
    start, end = ax.get_xlim()
    ax.set_title('area (1000 ha)')
    
    ax=axs[3]
    (icbm.loc[scn].groupby(['year']).sum().loc[:,'manure_cattle_kgC':]/1000000).plot(kind='area', color=['#eeeeee','#cccccc','#aaaaaa','#888888'], stacked=True, ax=ax)
    start, end = ax.get_xlim()
    ax.set_title('manure (1000 tonnes C)')
    plt.show()

In [ ]:
# Write csv-files
from datetime import date
for scn in session.scenarios():
    icbm.loc[[scn]].to_csv(f'{scn}_to_ICBM_{date.today().strftime("%y%m%d")}.csv')

In [ ]:
icbm.index.names, icbm.columns, str(icbm.index.get_level_values('scn').unique()[0])

# Code development section below

## Calculate tot SOC and CO2 in dataseries
code taken from old notebook
**TODO:** convert to operate with new class structure instead of dicts.

New code below:

Code to check instance attributes below:

In [4]:
FAI_soil.check_attributes_status()

('The following attributes are set for public variables',
 {'co2_fluxes': ('NoneType', False),
  'historic_ha_df': ('DataFrame', True),
  'historic_inventory': ('Dataset', True),
  'historic_sko_df': ('DataFrame', True),
  'input_inventory': ('Dataset', True),
  'name': ('str', True),
  'scenario': ('str', True),
  'soc_ha_df': ('DataFrame', True),
  'soc_inventory': ('Dataset', True),
  'soc_sko_df': ('DataFrame', True),
  'ss_input_df': ('DataFrame', True),
  'startyear': ('int32', True)})

In [5]:
FAI_soil.soc_inventory

<xarray.Dataset> Size: 368MB
Dimensions:      (region: 106, output_year: 100, input_year: 31, fraction: 14,
                  prod_system: 2, scn: 1)
Coordinates:
  * region       (region) object 848B '1011' '111' '1111' ... '911' '912' '913'
  * output_year  (output_year) datetime64[ns] 800B 2020-01-01 ... 2119-01-01
  * input_year   (input_year) datetime64[ns] 248B 2020-01-01 ... 2050-01-01
  * fraction     (fraction) object 112B 'i_ag_crop_ha' ... 'i_bg_crop_kgc'
  * prod_system  (prod_system) object 16B 'conventional' 'organic'
  * scn          (scn) object 8B 'fai'
Data variables:
    input_area   (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    y_pool       (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    o_pool       (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    tot_soc      (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    co2_flux     (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...

In [6]:
FAI_soil.soc_inventory.co2_flux

<xarray.DataArray 'co2_flux' (scn: 1, prod_system: 2, region: 106,
                              input_year: 31, output_year: 100, fraction: 14)> Size: 74MB
array([[[[[[            nan,             nan,             nan, ...,
                        nan,             nan,             nan],
           [-4.78254156e+03, -1.11130308e+08, -5.52882558e+02, ...,
            -1.22258790e+06, -2.53554414e+03, -5.89175853e+07],
           [ 1.27406567e+03,  2.96050350e+07,  1.18508860e+02, ...,
             2.62058363e+05,  5.04669220e+02,  1.17268287e+07],
           ...,
           [ 2.63570575e+00,  6.12450069e+04,  6.74690692e-01, ...,
             1.49194194e+03,  3.59322003e+00,  8.34944437e+04],
           [ 2.60948004e+00,  6.06356089e+04,  6.67977408e-01, ...,
             1.47709687e+03,  3.55746689e+00,  8.26636601e+04],
           [ 2.58351528e+00,  6.00322745e+04,  6.61330921e-01, ...,
             1.46239951e+03,  3.52206950e+00,  8.18411430e+04]],

          [[            nan,             nan,             nan, ...,
                        nan,             nan,             nan],
           [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, ...,
            -0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
           [-4.79547148e+03, -1.11372846e+08, -5.61722752e+02, ...,
            -1.20971251e+06, -2.55209381e+03, -5.92713257e+07],
...
           [ 2.27296125e+00,  5.00410085e+03,  3.26536355e+00, ...,
             1.38743161e+03,  4.83523864e+00,  1.06451537e+04],
           [ 2.25034490e+00,  4.95430920e+03,  3.23287263e+00, ...,
             1.37362643e+03,  4.78712721e+00,  1.05392326e+04],
           [ 2.22795359e+00,  4.90501299e+03,  3.20070501e+00, ...,
             1.35995862e+03,  4.73949450e+00,  1.04343655e+04]],

          [[            nan,             nan,             nan, ...,
                        nan,             nan,             nan],
           [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, ...,
            -0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
           [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, ...,
            -0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
           ...,
           [ 2.30793186e+00,  5.04682048e+03,  3.32269054e+00, ...,
             1.40831448e+03,  4.91039245e+00,  1.07376953e+04],
           [ 2.28496754e+00,  4.99660375e+03,  3.28962921e+00, ...,
             1.39430152e+03,  4.86153323e+00,  1.06308535e+04],
           [ 2.26223173e+00,  4.94688670e+03,  3.25689685e+00, ...,
             1.38042798e+03,  4.81316016e+00,  1.05250747e+04]]]]]])
Coordinates:
  * region       (region) object 848B '1011' '111' '1111' ... '911' '912' '913'
  * output_year  (output_year) datetime64[ns] 800B 2020-01-01 ... 2119-01-01
  * input_year   (input_year) datetime64[ns] 248B 2020-01-01 ... 2050-01-01
  * fraction     (fraction) object 112B 'i_ag_crop_ha' ... 'i_bg_crop_kgc'
  * prod_system  (prod_system) object 16B 'conventional' 'organic'
  * scn          (scn) object 8B 'fai'